# Download the Dinner Point Avenue profile

The profile is one line from the U.S. Geological Survey data release

Fiore, A. R. (2018). *Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014–15.* https://doi.org/10.5066/F7JH3KBD

The ScienceBase item is https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce. It holds ten MALA lines. Each line has a `.rd3` of samples, a `.rad` header, and an `.asc` text export of the same samples.

The folder `data/raw/usgs_forsythe/` keeps two files:

- `DinnerPointAve.rd3`, the samples
- `DinnerPointAve.rad`, the header for that line

A file that already matches the published size and MD5 checksum is left as it is. Any other `.rd3`, `.rad`, or `.asc` file in the folder is removed.


## Why a catalog request can come back empty

ScienceBase sits behind Cloudflare. A request from this notebook is often answered with HTTP 403 and an HTML page titled "Just a moment...", before the catalog JSON is sent. That page is a browser check, not a missing dataset. Opening the same item address in a browser returns the JSON.

The name, size, and MD5 below were copied from that catalog. They are what this notebook checks the two files against when the live catalog does not come through.


## Setup


In [1]:
import hashlib
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("open this notebook from the repository that contains data/raw")

ROOT = repo_root()
DEST = ROOT / "data" / "raw" / "usgs_forsythe"
ITEM = "https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce"
CATALOG = ITEM + "?format=json"
# Published name, size, and MD5 from the ScienceBase catalog.
NEEDED = [
    {
        "name": "DinnerPointAve.rad",
        "bytes": 776,
        "md5": "b3ba480372fc68eda6e7b9c539554ffe",
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__c9%2Ff1%2F2b%2Fc9f12ba52d4f5e227dc8c86b5bd4dd90e4629a12",
    },
    {
        "name": "DinnerPointAve.rd3",
        "bytes": 8518392,
        "md5": "9d248111ddef2b14ef9ca697acbf9671",
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__1a%2F05%2F89%2F1a05891cc21360a53e1b19ee11b7a13024b13399",
    },
]

def file_md5(path):
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

DEST.mkdir(parents=True, exist_ok=True)


## 1. Read the catalog when ScienceBase returns it

A JSON catalog replaces the saved size, checksum, and URL for the two files. A Cloudflare challenge leaves the saved values in place.


In [2]:
def read_catalog():
    request = Request(CATALOG, headers={"User-Agent": "GPRadar", "Accept": "application/json"})
    try:
        with urlopen(request, timeout=30) as response:
            payload = response.read()
    except HTTPError as error:
        payload = error.read()
        if error.code == 403 and b"Just a moment" in payload:
            print("ScienceBase did not return the catalog.")
            print("Cloudflare answered HTTP 403 with its browser check (Just a moment...) instead of the JSON.")
            print("The same item address in a browser returns the catalog. Using the saved name, size, and MD5.")
            return None
        raise
    if not payload.lstrip().startswith(b"{"):
        print("ScienceBase did not return JSON. Using the saved name, size, and MD5.")
        return None
    return json.loads(payload)

catalog = read_catalog()
if catalog is not None:
    by_name = {entry["name"]: entry for entry in catalog["files"]}
    for row in NEEDED:
        entry = by_name[row["name"]]
        row["bytes"] = int(entry["size"])
        row["url"] = entry["downloadUri"]
        checksum = entry.get("checksum") or {}
        if checksum.get("type") == "MD5" and checksum.get("value"):
            row["md5"] = checksum["value"]
    print("read the live ScienceBase catalog")

for row in NEEDED:
    print(f"{row['bytes']:>10} bytes  {row['md5']}  {row['name']}")


ScienceBase did not return the catalog.
Cloudflare answered HTTP 403 with its browser check (Just a moment...) instead of the JSON.
The same item address in a browser returns the catalog. Using the saved name, size, and MD5.
       776 bytes  b3ba480372fc68eda6e7b9c539554ffe  DinnerPointAve.rad
   8518392 bytes  9d248111ddef2b14ef9ca697acbf9671  DinnerPointAve.rd3


## 2. Check the two files, and download one that does not match

A file whose size and MD5 match the catalog is already the published copy. A missing or different file is downloaded. If that download is also the browser check, save the file from the item page into `data/raw/usgs_forsythe/` and run this cell again.


In [3]:
def download(row):
    path = DEST / row["name"]
    request = Request(row["url"], headers={"User-Agent": "GPRadar"})
    try:
        with urlopen(request, timeout=120) as response:
            payload = response.read()
    except HTTPError as error:
        payload = error.read()
        if error.code == 403 and b"Just a moment" in payload:
            raise RuntimeError(
                f"Cloudflare blocked the download of {row['name']}. "
                f"Save it from {ITEM} into {DEST}."
            ) from error
        raise
    if payload.lstrip()[:1] == b"<" or len(payload) != row["bytes"]:
        raise RuntimeError(
            f"The download of {row['name']} was not the published file "
            f"({len(payload)} bytes). Save it from {ITEM} into {DEST}."
        )
    path.write_bytes(payload)
    print(f"downloaded  {path.name}  ({len(payload)} bytes)")

for row in NEEDED:
    path = DEST / row["name"]
    if path.is_file() and path.stat().st_size == row["bytes"] and file_md5(path) == row["md5"]:
        print(f"matches the catalog  {path.name}  ({row['bytes']} bytes)")
        continue
    download(row)
    digest = file_md5(path)
    if digest != row["md5"]:
        raise RuntimeError(f"{path.name} MD5 is {digest}, catalog says {row['md5']}")
    print(f"checksum matches  {path.name}")


matches the catalog  DinnerPointAve.rad  (776 bytes)
matches the catalog  DinnerPointAve.rd3  (8518392 bytes)


## 3. Delete the other lines

Any other `.rd3`, `.rad`, or `.asc` file in the same folder is another line, or a text copy of a line. Those files are removed. `DinnerPointAve.rd3` and `DinnerPointAve.rad` stay.


In [4]:
keep = {row["name"] for row in NEEDED}
removed = []
for path in sorted(DEST.iterdir()):
    if path.is_file() and path.suffix.lower() in {".rd3", ".rad", ".asc"} and path.name not in keep:
        path.unlink()
        removed.append(path.name)

print(f"removed {len(removed)} files")
for name in removed:
    print(" ", name)
print("kept:")
for row in NEEDED:
    path = DEST / row["name"]
    print(f"  {path.name}  {path.stat().st_size} bytes  {file_md5(path)}")


removed 0 files
kept:
  DinnerPointAve.rad  776 bytes  b3ba480372fc68eda6e7b9c539554ffe
  DinnerPointAve.rd3  8518392 bytes  9d248111ddef2b14ef9ca697acbf9671
